# SRQS8F7JWA9MZ

In [ ]:
# Imports
from foodcast.imports import *
BASE_DIR, DATA_DIR_1, DATA_DIR_2, DATA_DIR_3, DATA_DIR_3_x = return_dir()

# Settings
notebook_settings() 

# Load Data
location_ids_by_coverage = load_loc_ids()
locations, before_after_details_true, items_tagged, customers = load_static()
loc_id = 'SRQS8F7JWA9MZ'
df_uncleaned = load_single_restaurant(loc_id)

In [ ]:
import yaml
# with open(Path('remapping') / 'loc1_remappings.yaml', "w", encoding="utf-8") as file:
#     yaml.dump(all_data, file, sort_keys=False, allow_unicode=True)
with open(Path('labeling') / 'remapping' / 'loc1_remappings.yaml', "r", encoding="utf-8") as f:
    config_data = yaml.load(f, Loader=yaml.FullLoader)

# Extract each key to its corresponding variable name
name_changes = config_data.get("name_changes", {})
modification_name_changes = config_data.get("modification_name_changes", [])
animal_product_conditions = config_data.get("animal_product_conditions", [])
alcoholic_drinks = config_data.get("alcoholic_drinks", [])
non_alcoholic_drinks = config_data.get("non_alcoholic_drinks", [])
merch_list = config_data.get("merch_list", [])
uncommon_list = config_data.get("uncommon_list", [])
rare_list = config_data.get("rare_list", [])
unknown_list = config_data.get("unknown_list", [])
remove_list = config_data.get("remove_list", [])
half_vegan_list = config_data.get("half_vegan_list", [])
vegan_list = config_data.get("vegan_list", [])
vegetarian_list = config_data.get("vegetarian_list", [])
meat_list = config_data.get("meat_list", [])

df_relabeled = fully_relabel_and_consolidate(
    df_uncleaned, 
    name_changes = name_changes,
    modification_name_changes = modification_name_changes, 
    vegan_list = vegan_list, 
    vegetarian_list = vegetarian_list, 
    meat_list = meat_list, 
    alcohol_list = alcoholic_drinks, 
    drinks_list = non_alcoholic_drinks, 
    merch = merch_list, 
    rare = rare_list + uncommon_list, 
    unknown = unknown_list, 
    remove_categories = ["Merch","Drink","Rare"]
    )

df_relabeled.to_parquet(DATA_DIR_3 / '1_relabeled' / (loc_id + '_sales_and_menu.parquet'))
# display(df_relabeled.groupby('item_name')['vegan'].unique()) # are the labels uniquely specified?

menu_changes = {
    'Gold Standard - Bacon' : [],
    'Gold Standard - Kale' : ["Vegan Gold Standard - Kale", "Meat Gold Standard - Kale"],
    'Gold Standard - Bacon & Kale' :[],
    'Gold Standard - Impossible' : ['Meat Gold Standard - Impossible', 'Vegan Gold Standard - Impossible'],
    'Beyond Burger' : ['Meat Beyond Burger', 'Vegan Beyond Burger'],
    'Impossible Patty Melt' : ['Meat Impossible Patty Melt', 'Vegan Impossible Patty Melt'],
    'The Alternative' : ['Meat The Alternative', 'Vegetarian The Alternative', 'Impossible The Alternative'],
    'Bottled Pop': df_relabeled.value_counts('item_name').filter(regex='Coca|Crod|Ginger|Soda').index.tolist(),
    'Canned Drinks': df_relabeled.value_counts('item_name').filter(regex='Coco|Mate|Zam|Croi').index.tolist(),
    }

df_consolidated = df_relabeled.pipe(rename_items, name_changes = menu_changes)
# display(df_consolidated.groupby('item_name')['vegan'].unique()) # there should be multiple labels for each item

plot_dish_time_series(df_consolidated, loc_id, before_after_details_true, legend=True, legend_label="Gold Standard - Impossible", legend_min=5.31, legend_max=12.75, shift_adjustment=0.02)

df_consolidated.to_parquet(DATA_DIR_3 / '2_consolidated' / (loc_id + '_sales_and_menu.parquet'))


In [ ]:
promo_date = before_after_details_true.loc[loc_id, 'cross_over_date']
print(df_uncleaned.query('item_name.str.contains("Impossible")')['item_quantity'].sum())
print(df_uncleaned.loc[promo_date:promo_date+pd.DateOffset(days=60)].query('item_name.str.contains("Impossible")')['item_quantity'].sum())

plot_time_series_subset(
                 df_uncleaned, 
                 exposure=promo_date,
                 freq='W', 
                 truncate=True)
plt.show()
plot_time_series_subset( 
                 df_uncleaned.query('item_name.str.contains("Impossible")'), 
                 exposure=promo_date, 
                 freq='W', 
                 truncate=False)
plt.show()

In [ ]:
# Price analysis
print(df_consolidated
      .query('~vegetarian')
      ['unit_price']
      .mean())

print((df_consolidated
       .query('~vegetarian')
       ['item_name']
       .nunique()) / (df_consolidated
                      ['item_name']
                      .nunique()))
plt.plot(df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['unit_price']
         .mean(), 
         df_consolidated
         .query('~vegetarian and dish_category != "Drink" and dish_category != "Alcohol"').resample('W')['item_name'].nunique() / df_consolidated.query('dish_category != "Drink" and dish_category != "Alcohol"')
         .resample('W')
         ['item_name']
         .nunique(), 
         'o', 
         alpha=0.5)
plt.show()